# PII Vault — Complete Guide

PII Vault detects and pseudonymises personally identifiable information in text before it reaches your LLM provider. This notebook covers:

- Tokenize & detokenize text
- Control which entity types to detect
- Format-preserving pseudonymisation
- LLM fallback for edge cases
- SafeOpenAI & SafeAnthropic drop-in proxies
- Persistent token storage (HostedVault)
- DSAR: list and delete by subject
- Batch processing
- Real-world: customer support pipeline

## Setup

In [ ]:
%pip install ag-pii-vault openai -q

In [ ]:
import os
AG_API_KEY = os.environ['AG_API_KEY']
# OPENAI_API_KEY = os.environ['OPENAI_API_KEY']  # needed for SafeOpenAI proxy

## 1. Basic tokenize

`tokenize_text` detects PII server-side and replaces it with format-preserving fakes. The original values are stored in the vault and can be retrieved later.

In [ ]:
from ag_pii_vault import HostedVault

vault = HostedVault(api_key=AG_API_KEY)

text = (
    'Hi, I am Sarah Connor. My email is sarah@skynet.com, '
    'phone +1-800-555-0199, SSN 123-45-6789, '
    'credit card 4111 1111 1111 1111, IP 192.168.1.42.'
)

result = vault.tokenize_text(text)
print('Original  :', text)
print()
print('Tokenized :', result['text'])
print()
print(f'Entities detected: {len(result["entities"])}')
for e in result['entities']:
    print(f'  {e["entity_type"]:<15} {e["original"]!r:30} → {e["replacement"]!r}')

## 2. Target specific entity types

Pass `entities` to restrict detection to a subset.

In [ ]:
# Only strip emails and phone numbers — leave everything else intact
result = vault.tokenize_text(text, entities=['EMAIL', 'PHONE'])
print(result['text'])

## 3. Detokenize — restore original values

In [ ]:
tokenized_text = result['text']

restored = vault.detokenize_text(tokenized_text)
print('Tokenized :', tokenized_text)
print('Restored  :', restored)

## 4. Format-preserving pseudonymisation

By default (`format_preserving=True`) replacements look like real values — fake emails still have `@`, fake phone numbers have the same digit pattern. Disable it to get opaque `[REDACTED_EMAIL]`-style tokens instead.

In [ ]:
# Opaque tokens
r_opaque = vault.tokenize_text(text, format_preserving=False)
print('Opaque:', r_opaque['text'][:100], '...')

## 5. Subject-scoped storage

Pass `subject_id` to link tokens to a user. This enables DSAR compliance — you can list or delete all PII for a given user with a single call.

In [ ]:
# Tokenize with a subject ID (e.g., your internal user ID)
result = vault.tokenize_text(text, subject_id='user_42')
print('Stored entries:', result['stored'])

In [ ]:
# List all tokens for this subject
entries = vault.list_subject('user_42')
print(f'{len(entries)} entries for user_42')
for e in entries[:3]:
    print(f'  {e["entity_type"]:<15} token={e["token"][:20]}...')

In [ ]:
# Delete all PII for this subject (GDPR right-to-erasure)
deleted = vault.delete_subject('user_42')
print(f'Deleted {deleted} entries for user_42')

## 6. Batch retrieve tokens

In [ ]:
# Re-tokenize for demo
r = vault.tokenize_text('Call me at +1-555-123-4567 or email me at alice@example.com')
tokens = [e['replacement'] for e in r['entities']]
print('Tokens:', tokens)

# Batch reverse lookup
values = vault.batch_retrieve(tokens)
for tok, val in values.items():
    print(f'  {tok[:30]}... → {val}')

## 7. Vault stats

In [ ]:
count = vault.count()
print(f'Total entries in vault: {count}')

## 8. SafeOpenAI proxy — zero-change integration

`SafeOpenAI` is a drop-in replacement for the OpenAI client. It strips PII from messages before sending and re-injects original values into the response — your OpenAI provider never sees raw PII.

In [ ]:
# from ag_pii_vault import SafeOpenAI, HostedVault
#
# vault = HostedVault(api_key=AG_API_KEY)
# client = SafeOpenAI(
#     api_key=os.environ['OPENAI_API_KEY'],
#     vault=vault,
# )
#
# response = client.chat.completions.create(
#     model='gpt-4o-mini',
#     messages=[{
#         'role': 'user',
#         'content': 'Summarise the support case for john@acme.com, phone +1-555-100-2000'
#     }],
# )
#
# # john@acme.com and +1-555-100-2000 never left your infrastructure
# print(response.choices[0].message.content)

## 9. SafeAnthropic proxy

In [ ]:
# from ag_pii_vault import SafeAnthropic, HostedVault
#
# vault = HostedVault(api_key=AG_API_KEY)
# client = SafeAnthropic(
#     api_key=os.environ['ANTHROPIC_API_KEY'],
#     vault=vault,
# )
#
# message = client.messages.create(
#     model='claude-3-5-haiku-20241022',
#     max_tokens=256,
#     messages=[{'role': 'user', 'content': 'Call Alice at alice@corp.com'}],
# )
# print(message.content[0].text)

## 10. Real-world: customer support pipeline

Strip PII from incoming tickets before sending to the LLM, then restore it in the final response.

In [ ]:
def handle_ticket(ticket_text: str, subject_id: str) -> str:
    """De-identify a support ticket, process it, re-identify the response."""
    # 1. Strip PII
    r = vault.tokenize_text(ticket_text, subject_id=subject_id)
    safe_text = r['text']
    print(f'Safe to send to LLM: {safe_text[:80]}...')

    # 2. Call your LLM (placeholder)
    llm_response = f'Thank you for contacting us. We have received your request for {safe_text[:40]}...'

    # 3. Restore PII in the response before returning to client
    final = vault.detokenize_text(llm_response)
    return final

ticket = 'Hi, I am Bob Lee (bob@example.com). My card 5500 0000 0000 0004 was charged incorrectly.'
response = handle_ticket(ticket, subject_id='customer_789')
print('\nFinal response:', response)

## Supported entity types

| Type | Example |
|------|---------|
| `EMAIL` | john@example.com |
| `PHONE` | +1-555-867-5309 |
| `SSN` | 123-45-6789 |
| `CREDIT_CARD` | 4111 1111 1111 1111 |
| `IP_ADDRESS` | 192.168.1.42 |
| `PERSON` | John Smith |
| `IBAN` | GB29 NWBK 6016 1331 9268 19 |
| `DATE_OF_BIRTH` | 1990-07-15 |
| `PASSPORT` | AB1234567 |
| `DRIVER_LICENSE` | D12345678 |

Full list: [ag-pii-vault README](https://github.com/aghribi/ag-pii-vault)